# Faster R-CNN (ResNet-50 v2) — UICVD / uicvd16
Thesis experiment: `fasterrcnn_r50_v2_uicvd_uicvd16`

**Datasets required (add via right panel):** `uidet-uicvd16`, `uidet-code`

**Secret required:** `WANDB_API_KEY`

In [ ]:
!pip install -q pycocotools wandb pyyaml

In [ ]:
import os, sys
from pathlib import Path

DATA_INPUT = Path('/kaggle/input/datasets/raresharitonov/uidet-uicvd16')
CODE_INPUT = Path('/kaggle/input/datasets/raresharitonov/uidet-code')
WORKING    = Path('/kaggle/working')

sys.path.insert(0, str(CODE_INPUT / 'src'))

print('Data input exists:', DATA_INPUT.exists())
print('Data contents:    ', sorted(p.name for p in DATA_INPUT.iterdir()))
print('Code src contents:', sorted(p.name for p in (CODE_INPUT / 'src').iterdir()))

import uidet
print('uidet imported OK:', uidet.__file__)

In [ ]:
import yaml, shutil

# Note: uidet-uicvd16.zip was built with the fixed script so no double-nesting
actual_images      = DATA_INPUT / 'images'
actual_annotations = DATA_INPUT / 'annotations'

print('images exists:     ', actual_images.exists())
print('annotations exists:', actual_annotations.exists())
print('annotation files:  ', sorted(f.name for f in actual_annotations.glob('*.json')))

work_data = WORKING / 'uicvd16'
work_data.mkdir(parents=True, exist_ok=True)

# Symlink images (large)
images_link = work_data / 'images'
if images_link.is_symlink(): images_link.unlink()
images_link.symlink_to(actual_images)
print('images link ok:', images_link.exists())

# Copy annotation JSONs (small - avoids symlink glob issues)
ann_dir = work_data / 'annotations'
if ann_dir.is_symlink(): ann_dir.unlink()
ann_dir.mkdir(exist_ok=True)
for f in actual_annotations.glob('*.json'):
    shutil.copy2(f, ann_dir / f.name)
    print('copied:', f.name)

# Write corrected data.yaml
with open(DATA_INPUT / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = str(work_data)
data_yaml_path = work_data / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print('data.yaml written OK')
print(data_cfg)

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    print('WandB logged in via Kaggle secret')
except Exception as e:
    print(f'Secret not found ({e}), falling back...')
    wandb.login()

In [ ]:
from uidet.models.base import TrainConfig, build_detector
from uidet.train import get_class_names

EXP_NAME   = 'fasterrcnn_r50_v2_uicvd_uicvd16'
MODEL_NAME = 'fasterrcnn_r50_v2'
DATASET    = 'uicvd'
TAXONOMY   = 'uicvd16'

out_dir = WORKING / 'results_v2' / EXP_NAME
out_dir.mkdir(parents=True, exist_ok=True)

class_names = get_class_names(data_yaml_path)
print(f'{len(class_names)} classes: {class_names}')

cfg = TrainConfig(
    name=EXP_NAME,
    out_dir=out_dir,
    data_yaml=data_yaml_path,
    coco_val_json=ann_dir / f'{DATASET}_{TAXONOMY}_val.json',
    coco_test_json=ann_dir / f'{DATASET}_{TAXONOMY}_test.json',
    epochs=50,
    batch=2,
    imgsz=640,
    lr0=0.0001,
    seed=42,
    device='0',
    workers=2,
    amp=True,
    extra={
        'min_size': 480, 'max_size': 800,
        'accumulate': 8, 'warmup_iters': 400,
        'weight_decay': 0.0005, 'patience': 8, 'min_epochs': 5,
    },
    wandb_project='uidet-thesis',
    wandb_entity=None,
    wandb_tags=['kaggle', 'fasterrcnn', DATASET, TAXONOMY],
)
print('Config OK ->', out_dir)

In [ ]:
import json
from uidet.train import _wandb_init, _wandb_log_summary, _wandb_finish

cfg_dict = {
    'name': EXP_NAME, 'model': MODEL_NAME,
    'dataset': DATASET, 'taxonomy': TAXONOMY,
    'wandb_project': 'uidet-thesis', 'wandb_entity': None, 'wandb_tags': ['kaggle'],
}
wandb_run = _wandb_init(cfg_dict, cfg, class_names)

detector = build_detector(MODEL_NAME, num_classes=len(class_names), class_names=class_names)
print(f'Training {MODEL_NAME} on {DATASET}/{TAXONOMY} ({len(class_names)} classes)...')

weights = detector.train(cfg)
print('Best weights:', weights)
# NOTE: faster_rcnn_tv.py already logs per-epoch data to WandB internally.
#       Do NOT call _wandb_log_epochs_from_csv here — it would try to re-log
#       the same steps and WandB would reject them all.

In [ ]:
import time
from uidet.utils.metrics import coco_evaluate, detections_to_coco, format_metrics_table

gt_path = ann_dir / f'{DATASET}_{TAXONOMY}_test.json'
gt      = json.loads(gt_path.read_text())
items   = [(im['id'], work_data / im['file_name']) for im in gt['images']]
name_to_cat_id = {c: i + 1 for i, c in enumerate(class_names)}

predictions, t0 = [], time.perf_counter()
for i in range(0, len(items), 4):
    chunk = items[i:i+4]
    batch_dets = detector.predict_batch([p for _, p in chunk], conf=0.001, iou=0.6)
    for image_id, dets in zip([iid for iid, _ in chunk], batch_dets):
        predictions.extend(detections_to_coco(dets, image_id, name_to_cat_id))
dt = time.perf_counter() - t0

eval_dir = out_dir / 'coco_eval_test'
metrics  = coco_evaluate(str(gt_path), predictions, eval_dir)
metrics['inference_seconds'] = dt
metrics['inference_fps']     = len(items) / dt
(eval_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(format_metrics_table(metrics))

In [ ]:
# Log COCO test metrics as a proper WandB chart point
if wandb_run is not None:
    wandb.log({
        'test/mAP':   metrics['mAP'],
        'test/mAP50': metrics['mAP50'],
        'test/mAP75': metrics['mAP75'],
        **{f'test/AP/{cls}':   v['AP']   for cls, v in metrics['per_class'].items()},
        **{f'test/AP50/{cls}': v['AP50'] for cls, v in metrics['per_class'].items()},
    })
    print('COCO metrics logged to WandB')

In [ ]:
_wandb_log_summary(wandb_run, metrics, split='test')
_wandb_finish(wandb_run)

import shutil
shutil.make_archive(str(WORKING / EXP_NAME), 'zip', str(out_dir))
print('Download:', str(WORKING / EXP_NAME) + '.zip')
print(json.dumps(metrics, indent=2))